In [100]:
# preprocessing (extracting the text and removing problematic characters)
# poetry run spacy download en_core_web_sm
# https://github.com/keredson/wordninja
from pypdf import PdfReader, mult
from nlp_text_clean.cleaner import TextCleaner
import wordninja
import re

reader = PdfReader("../text/eng-kjv_MAT.pdf")
cleaner = TextCleaner(
    remove_numbers=True,
    remove_punctuation=True,
    remove_special_chars=True,
    use_stemming=False,
    use_lemmatization=True,
    language="english",
    custom_stopwords=None,
    preserve_num_token=False,
    remove_html_tags=True,
    remove_urls=True,
)

HEADER_RE = re.compile(
    r"^\s*Matthew\s+\d+:\d+\s+[ivxlcdm]+\s+Matthew\s+\d+:\d+\s*",
    re.IGNORECASE | re.MULTILINE,
)

book = []
for i in range(len(reader.pages) - 1):

    # extracting the text from pdf
    text = reader.pages[i].extract_text()
    
    # strip header (matthew i matthew etc)
    text = HEADER_RE.sub("", text)     
    
    # cleaning text based on configs set in the instance
    cleaned_sentence = cleaner.clean_text(text)
    
    # necessary to separate joined words (caused by OCR, like yecome => ye come)
    sentence = ' '.join(wordninja.split(cleaned_sentence))

    book.append(sentence)

book

['gospel accord to st matthew book generation jesus christ son david son abraham abraham begat isaac isaac begat jacob jacob begat judas brother judas begat ph are zara th amar ph are begat es rom es rom begat aram aram begat amina dab amina dab begat naas son naas son begat salmon salmon begat booz racha b booz begat obe ruth obe begat jesse jesse begat david king david king begat solomon wife uri as solomon begat rob o am rob o am begat a bia a bia begat as a as a begat jos a phat jos a phat begat j or am j or am begat oz i as oz i as begat jo at ham jo at ham begat a chaz a chaz begat eze kia eze kia begat man asse man asse begat amon amon begat josias josias begat j echo nias brother time carry away babylon bring babylon j echo nias begat salat hi el salat hi el begat zoro babel',
 'zoro babel begat ab iud ab iud begat elia kim elia kim begat az or az or begat s a doc s a doc begat ac him ac him begat el iud el iud be get eleazar eleazar begat mat than mat than begat jacob jacob be

In [101]:
# tokenizing (making the text divided into even smaller chunks)
# https://www.nltk.org/api/nltk.tokenize.punkt.html
# https://www.nltk.org/
import nltk
from nltk.tokenize import word_tokenize

# downloading pre treined punkt model
nltk.download('punkt')

# tokeninzing text and adding to list
tokenized_book = []
for page in range(len(book)):
    tokenized_book.extend(word_tokenize(book[page]))

/home/arthur/Code/truth-seeker/.venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/arthur/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package punkt to /home/arthur/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [103]:
# https://www.geeksforgeeks.org/nlp/vectorization-techniques-in-nlp/
# https://www.geeksforgeeks.org/nlp/5-simple-ways-to-tokenize-text-in-python/
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

vectorizer = TfidfVectorizer(
    token_pattern=r"\S+$"
)
X = vectorizer.fit_transform(tokenized_book)

In [107]:
print(X.shape)                    # (n_verses, n_vocab)
print(vectorizer.get_feature_names_out()[:15])
print(vectorizer.idf_[:15])

print("Matrix shape:", X.shape)
print("Non-zero entries:", X.nnz)
print("Sparsity: {:.2%}".format(1 - X.nnz / (X.shape[0] * X.shape[1])))
print("Vocab size:", len(vectorizer.vocabulary_))

# What are the highest-IDF (rarest, most "informative") tokens?
idf = vectorizer.idf_
vocab = vectorizer.get_feature_names_out()
rare = vocab[np.argsort(idf)[-20:]]
print("Rarest tokens:", rare[::-1])

# What are the lowest-IDF (most common, least informative) tokens?
common = vocab[np.argsort(idf)[:20]]
print("Most common tokens:", common)

(11949, 1618)
['a' 'ab' 'abbas' 'abel' 'abide' 'ability' 'able' 'abode' 'abomination'
 'abound' 'abraham' 'abroad' 'abundance' 'ac' 'accord']
[6.8331385  9.28987427 8.59672709 9.69533938 9.69533938 9.69533938
 8.08590146 9.69533938 9.69533938 9.69533938 8.30904502 8.59672709
 9.0021922  9.28987427 8.59672709]
Matrix shape: (11949, 1618)
Non-zero entries: 11949
Sparsity: 99.94%
Vocab size: 1618
Rarest tokens: ['zara' 'zacharias' 'youth' 'yield' 'adjure' 'yes' 'guiltless' 'gross'
 'grievous' 'greeting' 'grape' 'grant' 'and' 'an' 'amar' 'alway'
 'alphaeus' 'womb' 'withdraw' 'with']
Most common tokens: ['unto' 'say' 'shall' 'ye' 'come' 'man' 'jesus' 'thou' 'go' 'e' 'the'
 'take' 'thy' 'see' 'son' 'one' 'th' 'lord' 'heaven' 'give']


In [108]:
from sklearn.metrics.pairwise import cosine_similarity

# Similarity between ALL pairs (n_verses × n_verses)
sim = cosine_similarity(X)

# Most similar verse to verse 0 (excluding itself)
def most_similar(i, sim, topn=5):
    scores = sim[i].copy()
    scores[i] = -1   # exclude self
    top = np.argsort(scores)[-topn:][::-1]
    return [(j, scores[j]) for j in top]

verse_texts = [" ".join(page) for page in tokenized_book if page]
for j, s in most_similar(0, sim):
    print(f"{s:.3f}  {verse_texts[j][:80]}")

def search(query, vectorizer, X, verse_texts, topn=5):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, X).ravel()
    top = np.argsort(scores)[-topn:][::-1]
    for j in top:
        print(f"{scores[j]:.3f}  {verse_texts[j][:90]}")

search("angel appear joseph dream", vectorizer, X, verse_texts)
# → should surface the annunciation + flight-to-Egypt verses

1.000  g o s p e l
1.000  g o s p e l
1.000  g o s p e l
1.000  g o s p e l
1.000  g o s p e l
1.000  d r e a m
1.000  d r e a m
1.000  d r e a m
1.000  d r e a m
1.000  d r e a m
